# HW3｜MCP 串接

本作業有**兩個任務**：

| | 在哪裡做 | 做什麼 |
|---|---|---|
| **任務一** | 你自己的電腦 | 用 MCP Inspector 連上 `everything`，截三張圖 |
| **任務二** | 這份 Colab | 讓模型呼叫同一台 server 取得資料，貼四段輸出 |

**兩個任務連的是同一台 MCP server**，差別在於「是誰決定要用那個工具」——
任務一是你自己按按鈕，任務二是模型自己決定。做完你就會看出差別在哪。

## 這份 notebook 不用寫程式

每一格都寫好了，你只要做三件事：

1. **第 4 格與第 7 格**：從五個候選問句裡挑一句，填進變數
2. **第 6 格**：填一個城市名稱
3. 其餘的格子按順序執行，把輸出貼進作答本

## 繳交

| 檔案 | 內容 |
|---|---|
| `HW3_學號_姓名.ipynb` | 這份 notebook，整份執行過、輸出留著 |
| `HW3_學號_姓名.docx` | 作答本，三張截圖與四段輸出都貼進去 |

**兩個檔案都上傳到 TronClass 本作業的繳交區，截止 W10（11/18）。**

> **先做任務一再做任務二。** 任務一的操作步驟在另一份「MCP Inspector
> 操作說明」裡（`HW3-Inspector操作說明.pdf`），照著做大約 40 分鐘。

---

## 任務一　在你自己的電腦上用 Inspector 連 `everything`

這一段**不在 Colab 做**——Inspector 需要 Node 22.19 以上，而 Colab 的 Node 固定是 20，換不了。請依照「MCP Inspector 操作說明」在自己的電腦上操作，把要求的畫面截圖貼進作答本。

### Inspector 連線成功畫面　→　請寫在**作答本第 2 題**

### Inspector 執行 get-sum　→　請寫在**作答本第 3 題**

### Inspector 的封包紀錄　→　請寫在**作答本第 4 題**

### 三種原物料各有幾個　→　請寫在**作答本第 5 題**

---

## 任務二　在 Colab 上讓模型取得資料

從這裡開始在 Colab 執行。請**依序**執行每一格。

### 第 1 格　裝套件

版本已經釘好，不要改。第一次執行約 1 分鐘。

In [ ]:
# 版本全部釘死——MCP 相關套件改版很快，不釘版會跑出跟作業說明不一樣的結果
!pip install -q "mcp==2.2.0" "nest_asyncio==1.6.0"

import importlib.metadata as M
for 套件 in ["mcp", "nest_asyncio", "transformers", "torch"]:
    print("%-14s %s" % (套件, M.version(套件)))

# Colab 的 Node 版本（Inspector 需要 22.19 以上，所以任務一不能在這裡做）
!node --version

In [ ]:
# ── 共用設定（不用改）──────────────────────────────────────────
import asyncio, json, os, re, sys, time
import nest_asyncio

# Colab 本身已經有一個在跑的事件迴圈，直接 asyncio.run() 會報
# RuntimeError: asyncio.run() cannot be called from a running event loop
nest_asyncio.apply()

from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client

# 本作業連的 MCP server：官方的參考實作，三種原物料都做了
EVERYTHING = "@modelcontextprotocol/server-everything@2026.8.31"


def 取(物件, *名稱, 預設="?"):
    """依序試幾個欄位名，回傳第一個存在的。

    這個 SDK 的欄位是 snake_case（server_info、is_error），
    但網路上的範例大量使用舊版的 camelCase（serverInfo、isError）。
    兩種都試，你照抄網路程式碼時比較不會踩到。
    """
    for n in 名稱:
        if hasattr(物件, n):
            return getattr(物件, n)
    return 預設


def 連(工作):
    """開一條連線、做事、關掉。

    errlog 一定要覆寫：Colab 的 sys.stderr 沒有 .fileno()，
    不覆寫會直接拋 UnsupportedOperation。這是 Colab 特有的坑，已經處理好。
    """
    async def _():
        params = StdioServerParameters(command="npx", args=["-y", EVERYTHING])
        with open(os.devnull, "w") as 空:
            async with stdio_client(params, errlog=空) as (讀, 寫):
                async with ClientSession(讀, 寫) as s:
                    await s.initialize()
                    return await 工作(s)
    return asyncio.run(_())


print("設定完成")

### 第 2 格　看這台 server 提供什麼

三種原物料各有幾個。**這三個數字應該跟你在 Inspector 上數到的一樣**（作答本第 5 題）。

In [ ]:
async def _清單(s):
    工具 = (await s.list_tools()).tools
    資源 = (await s.list_resources()).resources
    範本 = (await s.list_prompts()).prompts
    return 工具, 資源, 範本

t0 = time.time()
工具清單, 資源清單, 範本清單 = 連(_清單)
print("（連線耗時 %.1f 秒，第一次含下載）\n" % (time.time() - t0))

print("Tools     ", len(工具清單))
print("Resources ", len(資源清單))
print("Prompts   ", len(範本清單))

print("\n工具名稱與說明：")
for t in 工具清單:
    print("  %-30s %s" % (t.name, (t.description or "")[:48]))

### 第 3 格　載入模型

`Qwen2.5-3B-Instruct`，在 Colab 上跑，**不需要任何 API 金鑰**。

> ⚠ 請先確認上方選單 **執行階段 → 變更執行階段類型 → T4 GPU**，否則這一格會很慢或失敗。約 1–3 分鐘。

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

模型名 = "Qwen/Qwen2.5-3B-Instruct"

t0 = time.time()
斷詞器 = AutoTokenizer.from_pretrained(模型名)
模型 = AutoModelForCausalLM.from_pretrained(
    模型名,
    dtype=torch.float16,       # 新版 transformers 用 dtype，不是 torch_dtype
    device_map="auto",
)
模型.eval()
print("載入完成 %.0f 秒；裝置 = %s" % (time.time() - t0, 模型.device))

In [ ]:
# ── 把 MCP 的工具交給模型（不用改）──────────────────────────────
def 轉成模型用的工具(工具清單):
    """MCP 的工具描述直接當成模型的工具清單。

    注意送給模型的只有三樣東西：name、description、參數的 schema。
    **沒有函式主體**——模型是靠這幾行字判斷要不要用、怎麼用。
    """
    return [{
        "type": "function",
        "function": {
            "name": t.name,
            "description": t.description or "",
            "parameters": 取(t, "input_schema", "inputSchema", 預設={}) or {},
        },
    } for t in 工具清單]


抓呼叫 = re.compile(r"<tool_call>\s*(\{.*?\})\s*</tool_call>", re.S)


def 模型決定呼叫什麼(問句, 工具們):
    """把問句與工具清單交給模型，看它決定呼叫哪一個工具。"""
    提示 = 斷詞器.apply_chat_template(
        [{"role": "user", "content": 問句}],
        tools=工具們, add_generation_prompt=True, tokenize=False)
    輸入 = 斷詞器(提示, return_tensors="pt").to(模型.device)
    with torch.no_grad():
        產出 = 模型.generate(**輸入, max_new_tokens=256,
                             do_sample=False,       # 關掉隨機，同樣的問句結果一樣
                             pad_token_id=斷詞器.eos_token_id)
    回答 = 斷詞器.decode(產出[0][輸入["input_ids"].shape[1]:],
                         skip_special_tokens=True).strip()
    m = 抓呼叫.search(回答)
    if not m:
        # 模型也可能選擇不呼叫工具、直接回話——這是合法結果，照實記錄即可
        return None, None, 回答
    d = json.loads(m.group(1))
    return d.get("name"), d.get("arguments"), 回答


def 數token(工具們=None):
    """量一次「同樣一句話、有沒有附工具清單」各要幾個 token。"""
    額外 = {"tools": 工具們} if 工具們 else {}
    提示 = 斷詞器.apply_chat_template(
        [{"role": "user", "content": "x"}],
        add_generation_prompt=True, tokenize=False, **額外)
    return len(斷詞器(提示)["input_ids"])


工具們 = 轉成模型用的工具(工具清單)
print("交給模型的工具數 =", len(工具們))
print("工具說明佔掉的 token =", 數token(工具們) - 數token())
print("（這是每一次對話都要重複送出的固定成本）")

### Colab：模型自己呼叫工具

notebook 的**第 4 格**給了五個候選問句，挑一句填進去執行。
模型會自己判斷該呼叫哪一個工具、自己填參數。

**把那一格的輸出整段貼上**，然後填下表。

> 模型呼叫的工具跟你預期的不一樣，或是它根本沒有呼叫工具——
> **照實寫下來就好，不扣分。**

> **請把答案寫在作答本第 6 題**，不要寫在這一格。

In [ ]:
# ╔═══════════════════════════════════════════════════════════════╗
# ║  第 4 格　原物料一：Tools —— 由「模型」決定要不要用               ║
# ╚═══════════════════════════════════════════════════════════════╝
#
# 從下面五句挑一句，把代號填進 我的選擇。
#
#   A　3 加 5 等於多少？
#   B　幫我算 128 加 457
#   C　幫我把這句話原封不動重複一次：MCP 很有趣
#   D　請重複這段文字：輔仁大學資訊管理學系
#   E　24 和 18 加起來是多少？

我的選擇 = "A"          # ← 改成 A / B / C / D / E

候選 = {
    "A": "3 加 5 等於多少？",
    "B": "幫我算 128 加 457",
    "C": "幫我把這句話原封不動重複一次：MCP 很有趣",
    "D": "請重複這段文字：輔仁大學資訊管理學系",
    "E": "24 和 18 加起來是多少？",
}
問句 = 候選[我的選擇]
print("我的問句（%s）：%s\n" % (我的選擇, 問句))

# 第一步：模型看著 13 個工具的說明，自己決定要呼叫哪一個
工具名, 參數, 原話 = 模型決定呼叫什麼(問句, 工具們)

if 工具名 is None:
    print("模型沒有呼叫任何工具，它直接回答了：")
    print(" ", 原話)
    print("\n→ 這也是合法的結果，照實寫進作答本第 6 題就好。")
else:
    print("模型決定呼叫的工具 :", 工具名)
    print("模型填的參數       :", json.dumps(參數, ensure_ascii=False))

    # 第二步：真的去呼叫它
    async def _做(s):
        return await s.call_tool(工具名, 參數)
    回傳 = 連(_做)

    print("是否出錯           :", 取(回傳, "is_error", "isError"))
    print("回傳內容           :")
    for c in 回傳.content:
        print("  ", getattr(c, "text", "[%s]" % getattr(c, "type", "?")))

### Colab：程式讀取資源

notebook 的**第 5 格**不經過模型——是**程式直接去讀**一份文件，
再把內容交給模型回答問題。

**把那一格的輸出整段貼上**，然後填下表。

> **請把答案寫在作答本第 7 題**，不要寫在這一格。

In [ ]:
# ╔═══════════════════════════════════════════════════════════════╗
# ║  第 5 格　原物料二：Resources —— 由「程式」去讀                  ║
# ╚═══════════════════════════════════════════════════════════════╝
#
# 這一格完全不經過模型挑選：資源是唯讀資料，要由應用程式自己
# 用 URI 去讀。讀回來之後才交給模型當作回答的依據。
# 這一格不用改任何東西，直接執行。

async def _讀資源(s):
    清單 = (await s.list_resources()).resources
    目標 = 清單[0].uri                      # 取第一份文件
    內容 = await s.read_resource(目標)
    return 目標, 內容

URI, 資源內容 = 連(_讀資源)

print("程式讀取的 URI :", URI)
全文 = ""
for c in 資源內容.contents:
    print("mime type      :", 取(c, "mime_type", "mimeType"))
    全文 += getattr(c, "text", "") or ""

前兩行 = [l for l in 全文.splitlines() if l.strip()][:2]
print("\n文件內容的前兩行：")
for l in 前兩行:
    print("  ", l)

# 把讀回來的文件交給模型，讓它依據這份文件回答
提問 = "根據下面這份文件，用一句話說明它在講什麼：\n\n" + 全文[:1500]
提示 = 斷詞器.apply_chat_template([{"role": "user", "content": 提問}],
                                  add_generation_prompt=True, tokenize=False)
輸入 = 斷詞器(提示, return_tensors="pt").to(模型.device)
with torch.no_grad():
    產出 = 模型.generate(**輸入, max_new_tokens=128, do_sample=False,
                         pad_token_id=斷詞器.eos_token_id)
print("\n模型依據這份文件的回答：")
print(" ", 斷詞器.decode(產出[0][輸入["input_ids"].shape[1]:],
                         skip_special_tokens=True).strip())

### Colab：使用者挑選提示詞範本

notebook 的**第 6 格**要你自己挑一個城市名稱填進去，
然後去把 `args-prompt` 這個範本取回來。

**把那一格的輸出整段貼上**，然後填下表。
最後一欄請注意看：你填的那個字，出現在展開後的句子的哪個位置？

> **請把答案寫在作答本第 8 題**，不要寫在這一格。

In [ ]:
# ╔═══════════════════════════════════════════════════════════════╗
# ║  第 6 格　原物料三：Prompts —— 由「你」挑選                      ║
# ╚═══════════════════════════════════════════════════════════════╝
#
# 提示詞範本是伺服器先寫好的句型，由使用者挑選、並在執行時填入參數。
# 這台 server 的 args-prompt 需要一個參數 city。
#
# 從下面五個挑一個填進去（也可以填別的城市）：
#   台北 / 京都 / 紐約 / 巴黎 / 新北

我填的城市 = "台北"        # ← 改成你要的城市

async def _取範本(s):
    return await s.get_prompt("args-prompt", {"city": 我填的城市})

範本 = 連(_取範本)

print("我填的參數 city =", 我填的城市)
print("\n範本展開後的內容：")
for m in 範本.messages:
    print("  role =", m.role)
    print("  text =", getattr(m.content, "text", "（非文字內容）"))

print("\n→ 注意你填的那個字出現在句子的哪個位置（作答本第 8 題）")

### Colab：取得圖片或資源連結

notebook 的**第 7 格**同樣給了五個候選問句，挑一句填進去執行。

這一格要看的是：**MCP 回傳的不一定是文字。**
注意輸出裡的「區塊」——一次回傳可能同時包含文字、圖片或資源連結。

> **請把答案寫在作答本第 9 題**，不要寫在這一格。

In [ ]:
# ╔═══════════════════════════════════════════════════════════════╗
# ║  第 7 格　MCP 回傳的不一定是文字                                ║
# ╚═══════════════════════════════════════════════════════════════╝
#
# 前面三格拿回來的都是文字。這一格要看的是：一次回傳可以同時包含
# 文字、圖片、資源連結等不同型別的「區塊」。
#
# 從下面五句挑一句，把代號填進 我的選擇2。
#
#   A　給我看一下 MCP 的 logo 圖
#   B　給我 3 個資源連結
#   C　幫我拿一張圖片
#   D　列出幾個資源連結給我
#   E　我想看那張小圖

我的選擇2 = "A"         # ← 改成 A / B / C / D / E

候選2 = {
    "A": "給我看一下 MCP 的 logo 圖",
    "B": "給我 3 個資源連結",
    "C": "幫我拿一張圖片",
    "D": "列出幾個資源連結給我",
    "E": "我想看那張小圖",
}
問句2 = 候選2[我的選擇2]
print("我的問句（%s）：%s\n" % (我的選擇2, 問句2))

工具名2, 參數2, 原話2 = 模型決定呼叫什麼(問句2, 工具們)

if 工具名2 is None:
    print("模型沒有呼叫任何工具，它直接回答了：")
    print(" ", 原話2)
    print("\n→ 照實寫進作答本第 9 題就好。")
else:
    print("模型決定呼叫的工具 :", 工具名2)
    print("模型填的參數       :", json.dumps(參數2, ensure_ascii=False))

    async def _做2(s):
        return await s.call_tool(工具名2, 參數2 or {})
    回傳2 = 連(_做2)

    print("是否出錯           :", 取(回傳2, "is_error", "isError"))
    print("回傳一共有 %d 個區塊：" % len(回傳2.content))
    for i, c in enumerate(回傳2.content, 1):
        型別 = getattr(c, "type", "?")
        文 = getattr(c, "text", None)
        if 文 is not None:
            print("  區塊 %d　型別 = %-14s 內容 = %s" % (i, 型別, 文[:60]))
        else:
            print("  區塊 %d　型別 = %-14s mime = %s（不是文字）"
                  % (i, 型別, 取(c, "mime_type", "mimeType")))

### 第 8 格　三種原物料的差別

執行這一格，把它印出來的表看一次，然後回答作答本第 10 題。

In [ ]:
# 把前面四格做過的事並排，差別就看得出來了
print("你在這份 notebook 做過的四件事：\n")
print("  第 4 格　Tools      　你只問了一句話，工具是模型自己挑的")
print("  第 5 格　Resources  　沒有經過模型，是程式用 URI 直接去讀")
print("  第 6 格　Prompts    　範本是你自己挑的，參數也是你填的")
print("  第 7 格　（還是 Tools）回傳裡同時有文字與非文字的區塊")
print()
print("→ 同一台 server 上的三種原物料，發動的人不一樣。")
print("  請把「是誰決定要用它」填進作答本第 10 題。")

### 三種原物料分別由誰決定　→　請寫在**作答本第 10 題**

### 名詞定義　→　請寫在**作答本第 1 題**

---

## 繳交前檢查

- [ ] 這份 notebook「重新啟動並全部執行」過一次，沒有出錯
- [ ] 作答本第 2、3、4 題貼了三張 Inspector 截圖
- [ ] 作答本第 6、7、8、9 題貼了四段輸出
- [ ] 兩個檔名都改成 `HW3_學號_姓名`

> 模型呼叫的工具跟你預期的不一樣，或它根本沒有呼叫工具——**照實寫，不扣分。**